# Extreme Gradient Boosting

In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import AdaBoostClassifier,GradientBoostingClassifier
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import make_column_selector, make_column_transformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler, MaxAbsScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from tqdm import tqdm
from xgboost import XGBClassifier

In [3]:
df = pd.read_csv("../Day5/HR_comma_sep.csv")

In [4]:
df.head()

,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,Department,salary
0,0.38,0.53,2,157,3,0,1,0,sales,low
1,0.80,0.86,5,262,6,0,1,0,sales,medium
2,0.10,0.77,6,247,4,0,1,0,sales,low
3,0.92,0.85,5,259,5,0,1,0,sales,low
4,0.89,1.00,5,224,5,0,1,0,sales,low


In [5]:
X,y = df.drop("left",axis=1), df["left"]
X_train, X_test, y_train, y_test = train_test_split(X,y,random_state=26,test_size=0.3,stratify=y)

In [8]:
ohc = OneHotEncoder(sparse_output=False, drop="first").set_output(transform="pandas")
ct = make_column_transformer((ohc,make_column_selector(dtype_include=object)),
                             ("passthrough", make_column_selector(dtype_include=['int64','float64'])),
                             verbose_feature_names_out=False)
ct = ct.set_output(transform='pandas')

## GradientBoostingClassifier

In [10]:
lr = np.linspace(0.01,0.9,10)
max_depth = [None, 2, 3, 4]
n_estimators = [25, 50, 100]
scores = []
for l in tqdm(lr):
    for m in max_depth:
        for n in n_estimators:
            gdc = GradientBoostingClassifier(random_state=26,learning_rate=l, n_estimators= n, max_depth= m)
            pipe = Pipeline([("CT",ct),("GDC",gdc)])
            pipe.fit(X_train,y_train)
            y_pred = pipe.predict(X_test)
            scores.append([l,m,n,accuracy_score(y_test, y_pred)])
df_scores = pd.DataFrame(scores, columns=["Learning Rate","Max Depth", "N", "Score"])
df_scores.sort_values("Score",ascending=False)

100%|██████████████████████████████████████████████████████████████████████████████████| 10/10 [02:20<00:00, 14.02s/it]


,Learning Rate,Max Depth,N,Score
83,0.603333,4.0,100,0.987553
71,0.504444,4.0,100,0.985552
82,0.603333,4.0,50,0.983774
1,0.010000,NaN,50,0.983552
59,0.405556,4.0,100,0.983330
...,...,...,...,...
4,0.010000,2.0,50,0.762392
9,0.010000,4.0,25,0.762169
6,0.010000,3.0,25,0.762169
3,0.010000,2.0,25,0.762169


# XGBoostClassifier

In [11]:
lr = np.linspace(0.01,0.9,10)
max_depth = [None, 2, 3, 4]
n_estimators = [25, 50, 100]
scores = []
for l in tqdm(lr):
    for m in max_depth:
        for n in n_estimators:
            gdc = XGBClassifier(random_state=26,learning_rate=l, n_estimators= n, max_depth= m)
            pipe = Pipeline([("CT",ct),("GDC",gdc)])
            pipe.fit(X_train,y_train)
            y_pred = pipe.predict(X_test)
            scores.append([l,m,n,accuracy_score(y_test, y_pred)])
df_scores = pd.DataFrame(scores, columns=["Learning Rate","Max Depth", "N", "Score"])
df_scores.sort_values("Score",ascending=False)

100%|██████████████████████████████████████████████████████████████████████████████████| 10/10 [00:22<00:00,  2.20s/it]


,Learning Rate,Max Depth,N,Score
62,0.504444,NaN,100,0.988220
38,0.306667,NaN,100,0.987997
85,0.702222,NaN,50,0.987775
73,0.603333,NaN,50,0.987775
109,0.900000,NaN,50,0.987553
...,...,...,...,...
4,0.010000,2.0,50,0.762392
3,0.010000,2.0,25,0.762169
6,0.010000,3.0,25,0.762169
9,0.010000,4.0,25,0.762169
